# CONUS HumanET × SIF × Drought Analysis

**HumanET** = OpenET − NLDAS Noah ET isolates anthropogenic water additions (primarily irrigation). This notebook extends the Iowa analysis to all of CONUS for 2015–2024.

**Prerequisites**: Run all download scripts (06, 16–19) then `00_align_spatial_data_conus.ipynb` before this notebook.

**Approach**: Continuous GRIDMET drought indices (SPI, SPEI, EDDI) are binned into USDM-equivalent categories for comparison with SIF anomalies across irrigated vs rainfed cropland.

In [1]:
import sys, os, gc
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import rasterio
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
from datetime import date

_root_env    = os.environ.get('SIF_ROOT')
project_root = Path(_root_env) if _root_env else Path('../../..').resolve()

proc  = project_root / 'data' / 'processed' / 'conus'
raw   = project_root / 'data' / 'raw'
figs  = project_root / 'figures' / 'conus'
figs.mkdir(parents=True, exist_ok=True)

# Input subdirs
cdl_proc    = proc / 'cdl'
openet_proc = proc / 'openet'
nldas_proc  = proc / 'nldas'
ndvi_proc   = proc / 'ndvi'
sif_proc    = proc / 'sif'
usdm_proc   = proc / 'drought_usdm'
gmet_proc   = proc / 'drought_gridmet'

# CONUS NLDAS grid
CONUS_LON = np.arange(-124.6875, -84.0625, 0.125)   # 325 — matches processed data grid
CONUS_LAT = np.arange( 49.3125,  25.6875, -0.125)   # 189 — matches processed data grid
YEARS     = list(range(2015, 2025))
MONTHS    = list(range(1, 13))

print('Project root:', project_root)
print('CONUS grid: 325 x 189 at 0.125 deg (cropped CONUS ag region)')


Project root: /home/pielab-sandbox-jcoldiron/SIF-Analysis
CONUS grid: 325 x 189 at 0.125 deg (cropped CONUS ag region)


## 1. Configuration & Thresholds

In [2]:
# -- Irrigation threshold -------------------------------------------------------
IRR_THRESHOLD_MM = 20.0   # mm/month: HumanET > 20 = high-confidence irrigation

# -- Cropland masking ----------------------------------------------------------
CDL_CROP_FRAC_MIN = 0.5   # >=50% of cell must be cropland to include

# -- Growing season months -----------------------------------------------------
GROWING_SEASON = [4, 5, 6, 7, 8, 9]   # April-September

# -- Drought categorization from GRIDMET indices -------------------------------
# Following USDM-equivalent thresholds (Svoboda et al. 2002; WMO 2012)
# Category: -1=No drought, 0=D0 (Abnormally Dry), 1=D1, 2=D2, 3=D3, 4=D4
DM_LABELS  = {-1: 'No drought', 0: 'D0', 1: 'D1', 2: 'D2', 3: 'D3', 4: 'D4'}
DM_COLORS  = {-1: '#A8D5A2', 0: '#FFFF00', 1: '#F5BE4E', 2: '#E08C32', 3: '#C74B2A', 4: '#6B0F0F'}

def spei_to_dm(spei):
    """Bin SPEI/SPI values into USDM-equivalent DM categories (-1 to 4)."""
    dm = np.full_like(spei, -1, dtype=float)  # default: no drought
    dm = np.where(spei < -0.5,  0, dm)   # D0
    dm = np.where(spei < -1.0,  1, dm)   # D1
    dm = np.where(spei < -1.5,  2, dm)   # D2
    dm = np.where(spei < -2.0,  3, dm)   # D3
    dm = np.where(spei < -2.5,  4, dm)   # D4
    return dm

def eddi_to_dm(eddi):
    """Bin EDDI values into DM categories. EDDI is positive for dry conditions."""
    dm = np.full_like(eddi, -1, dtype=float)
    dm = np.where(eddi >  0.5,  0, dm)
    dm = np.where(eddi >  1.0,  1, dm)
    dm = np.where(eddi >  1.5,  2, dm)
    dm = np.where(eddi >  2.0,  3, dm)
    dm = np.where(eddi >  2.5,  4, dm)
    return dm

def pdsi_to_dm(pdsi):
    """Bin PDSI into DM categories."""
    dm = np.full_like(pdsi, -1, dtype=float)
    dm = np.where(pdsi < -1.0,  0, dm)
    dm = np.where(pdsi < -2.0,  1, dm)
    dm = np.where(pdsi < -3.0,  2, dm)
    dm = np.where(pdsi < -4.0,  3, dm)
    dm = np.where(pdsi < -5.0,  4, dm)
    return dm

print('Drought thresholds configured.')
print('SPI/SPEI: D0 < -0.5, D1 < -1.0, D2 < -1.5, D3 < -2.0, D4 < -2.5')
print('EDDI:     D0 >  0.5, D1 >  1.0, D2 >  1.5, D3 >  2.0, D4 >  2.5')
print('PDSI:     D0 < -1.0, D1 < -2.0, D2 < -3.0, D3 < -4.0, D4 < -5.0')

Drought thresholds configured.
SPI/SPEI: D0 < -0.5, D1 < -1.0, D2 < -1.5, D3 < -2.0, D4 < -2.5
EDDI:     D0 >  0.5, D1 >  1.0, D2 >  1.5, D3 >  2.0, D4 >  2.5
PDSI:     D0 < -1.0, D1 < -2.0, D2 < -3.0, D3 < -4.0, D4 < -5.0


## 2. CDL Cropland Mask

Build a spatially explicit cropland mask from CDL fraction rasters. Cells with ≥50% cropland coverage are included in the analysis. Crop-type fractions (corn, soy, wheat) enable crop-specific sub-analyses.

In [3]:
# Load CDL multi-band fraction rasters.
#
# File format: CDL_CONUS_{year}.tif — 6-band float32 raster on the 325x189 grid
#   Band 1: cropland_frac   Band 2: corn_frac   Band 3: soy_frac
#   Band 4: wheat_frac      Band 5: cotton_frac  Band 6: pasture_frac
#
# All source datasets (CDL, OpenET, NLDAS) share the same 325x189, 0.125-degree
# CONUS grid, so bands are read directly without reprojection.

n_lat, n_lon = len(CONUS_LAT), len(CONUS_LON)

cropland_frac_all = np.full((len(YEARS), n_lat, n_lon), np.nan)
corn_frac_all     = np.full((len(YEARS), n_lat, n_lon), np.nan)
soy_frac_all      = np.full((len(YEARS), n_lat, n_lon), np.nan)

cdl_missing = []

for yi, year in enumerate(YEARS):
    fp = cdl_proc / f'CDL_CONUS_{year}.tif'
    if not fp.exists():
        cdl_missing.append(year)
        continue

    with rasterio.open(fp) as src:
        # Discover which band index holds each fraction by reading band-level tags
        band_map = {}
        for b in range(1, src.count + 1):
            name = src.tags(b).get('name', f'band{b}')
            band_map[name] = b

        for arr_store, band_name, default_band in [
            (cropland_frac_all, 'cropland_frac', 1),
            (corn_frac_all,     'corn_frac',     2),
            (soy_frac_all,      'soy_frac',      3),
        ]:
            b_idx = band_map.get(band_name, default_band)
            arr = src.read(b_idx).astype(float)
            nd = src.nodata
            if nd is not None:
                arr[arr == nd] = np.nan
            if arr.shape == (n_lat, n_lon):
                arr_store[yi] = arr
            else:
                print(f'  WARNING: CDL {year} shape {arr.shape} != ({n_lat},{n_lon})')

if cdl_missing:
    print('Missing CDL years:', cdl_missing)

# Mean fraction across all available years
cropland_frac_mean = np.nanmean(cropland_frac_all, axis=0)
corn_frac_mean     = np.nanmean(corn_frac_all,     axis=0)
soy_frac_mean      = np.nanmean(soy_frac_all,      axis=0)

# Static cropland mask: cells with >= 50% mean cropland fraction
crop_mask_static = (cropland_frac_mean >= CDL_CROP_FRAC_MIN)

n_crop_cells = crop_mask_static.sum()
n_total      = crop_mask_static.size
print('Static cropland mask (>=50%) covers', n_crop_cells, 'of', n_total, 'CONUS cells')
print('Cropland fraction of CONUS: ' + str(round(100 * n_crop_cells / n_total, 1)) + '%')
if n_crop_cells > 0:
    print('Mean cropland_frac inside mask:', round(float(np.nanmean(cropland_frac_mean[crop_mask_static])), 3))
    print('Mean corn_frac inside mask:    ', round(float(np.nanmean(corn_frac_mean[crop_mask_static])), 3))
    print('Mean soy_frac inside mask:     ', round(float(np.nanmean(soy_frac_mean[crop_mask_static])), 3))


Static cropland mask (>=50%) covers 25153 of 61425 CONUS cells
Cropland fraction of CONUS: 40.9%
Mean cropland_frac inside mask: 0.911
Mean corn_frac inside mask:     0.077
Mean soy_frac inside mask:      0.071


In [4]:
if True:
    try:
        fig, axes = plt.subplots(1, 2, figsize=(18, 5))

        # Panel 1: mean cropland fraction
        ax = axes[0]
        im0 = ax.imshow(
            cropland_frac_mean,
            extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
            origin='upper', cmap='YlGn', vmin=0, vmax=1, aspect='auto'
        )
        plt.colorbar(im0, ax=ax, label='Cropland fraction')
        ax.set_title('Mean Cropland Fraction (CDL 2015\u20132024)', fontsize=12)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.contour(
            CONUS_LON, CONUS_LAT, crop_mask_static.astype(float),
            levels=[0.5], colors='k', linewidths=0.5, alpha=0.5
        )

        # Panel 2: corn+soy combined fraction
        ax = axes[1]
        cornsoy = np.nansum([corn_frac_mean, soy_frac_mean], axis=0)
        cornsoy[cornsoy == 0] = np.nan
        im1 = ax.imshow(
            cornsoy,
            extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
            origin='upper', cmap='BuGn', vmin=0, vmax=1, aspect='auto'
        )
        plt.colorbar(im1, ax=ax, label='Corn + Soy fraction')
        ax.set_title('Corn + Soy Fraction (CDL 2015\u20132024)', fontsize=12)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')

        plt.tight_layout()
        plt.savefig(figs / 'conus_cdl_fraction_mean.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved conus_cdl_fraction_mean.png')
    except Exception as e:
        print('CDL map skipped:', e)

Saved conus_cdl_fraction_mean.png


## 3. HumanET = OpenET − NLDAS Noah ET

Computes the HumanET signal for each month and grid cell. Positive values indicate anthropogenic water additions (primarily irrigation). Threshold: **>20 mm/month** = high-confidence irrigation.

In [5]:
# Load all OpenET and NLDAS files, compute delta, mask to cropland
# Store as monthly arrays: shape (n_months, 224, 464)
# n_months = 120 (2015-2024)

records    = []
delta_maps = {}   # key: (year, month) -> 2D array

n_proc = 0
n_fail = 0

for year in YEARS:
    # Per-year cropland mask (use year-specific CDL if available)
    yi = YEARS.index(year)
    year_mask = (cropland_frac_all[yi] >= CDL_CROP_FRAC_MIN)
    if year_mask.sum() == 0:
        year_mask = crop_mask_static   # fallback to static mask

    for month in MONTHS:
        yyyymm = f'{year}{month:02d}'

        fp_openet = openet_proc / f'OpenET_CONUS_{yyyymm}.tif'
        fp_nldas  = nldas_proc  / f'NLDAS_Evap_{yyyymm}.nc'

        openet_arr = None
        nldas_arr  = None

        # Load OpenET
        if fp_openet.exists():
            try:
                with rasterio.open(fp_openet) as src:
                    a = src.read(1).astype(float)
                    nd = src.nodata
                    if nd is not None:
                        a[a == nd] = np.nan
                    if a.shape == (n_lat, n_lon):
                        openet_arr = a
            except Exception as e:
                print(f'  OpenET read error {yyyymm}: {e}')

        # Load NLDAS Noah ET
        if fp_nldas.exists():
            try:
                ds = xr.open_dataset(fp_nldas)
                # Variable name may be 'Evap', 'et', 'EVP', or similar
                et_var = None
                for vname in ['Evap', 'EVP', 'et', 'ET', 'aevap_surface']:
                    if vname in ds:
                        et_var = vname
                        break
                if et_var is None:
                    et_var = list(ds.data_vars)[0]
                arr_n = ds[et_var].values
                ds.close()
                if arr_n.ndim == 3:
                    arr_n = arr_n[0]
                if arr_n.shape == (n_lat, n_lon):
                    nldas_arr = arr_n.astype(float)
                elif arr_n.shape == (n_lon, n_lat):
                    nldas_arr = arr_n.T.astype(float)
            except Exception as e:
                print(f'  NLDAS read error {yyyymm}: {e}')

        if openet_arr is not None and nldas_arr is not None:
            delta = openet_arr - nldas_arr
            delta_masked = np.where(year_mask, delta, np.nan)

            # Mask out extreme outliers (>500 mm/month)
            delta_masked = np.where(np.abs(delta_masked) > 500, np.nan, delta_masked)

            valid     = ~np.isnan(delta_masked)
            irr_cells = (delta_masked > IRR_THRESHOLD_MM) & valid
            n_valid   = valid.sum()
            irr_frac  = irr_cells.sum() / n_valid if n_valid > 0 else np.nan
            mean_d    = float(np.nanmean(delta_masked))

            delta_maps[(year, month)] = delta_masked
            records.append({
                'year':     year,
                'month':    month,
                'yyyymm':   yyyymm,
                'mean_delta': mean_d,
                'irr_frac':   irr_frac,
                'n_cells':    n_valid
            })
            n_proc += 1
        else:
            records.append({
                'year': year, 'month': month, 'yyyymm': yyyymm,
                'mean_delta': np.nan, 'irr_frac': np.nan, 'n_cells': 0
            })
            n_fail += 1

df_et = pd.DataFrame(records)
df_et['date'] = pd.to_datetime(df_et['yyyymm'], format='%Y%m')

print('Months processed:', n_proc)
print('Months missing:  ', n_fail)
print()
print('Mean HumanET by month (growing season):')
gs_df = df_et[df_et['month'].isin(GROWING_SEASON)]
print(gs_df.groupby('month')['mean_delta'].mean().round(2).to_string())

Months processed: 120
Months missing:   0

Mean HumanET by month (growing season):
month
4    17.83
5    17.13
6    21.45
7    30.63
8    27.94
9    19.12


In [6]:
if True:
    try:
        fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=True)

        # Top: monthly mean HumanET bar chart
        ax = axes[0]
        colors_bar = [
            '#2166ac' if v > IRR_THRESHOLD_MM else '#d1e5f0'
            for v in df_et['mean_delta']
        ]
        ax.bar(df_et['date'], df_et['mean_delta'], color=colors_bar, width=25, align='center')
        ax.axhline(IRR_THRESHOLD_MM, color='red', linestyle='--', linewidth=1,
                   label=f'Irrigation threshold ({IRR_THRESHOLD_MM} mm/mo)')
        ax.axhline(0, color='k', linewidth=0.7)
        ax.set_ylabel('HumanET (mm/month)', fontsize=11)
        ax.set_title('CONUS Cropland HumanET (OpenET \u2212 NLDAS Noah ET), 2015\u20132024', fontsize=13)
        ax.legend(fontsize=9)
        ax.set_ylim(-30, 120)

        # Bottom: fraction of irrigated cropland cells
        ax = axes[1]
        ax.fill_between(df_et['date'], df_et['irr_frac'] * 100, color='#2166ac', alpha=0.7,
                        label='Irrigated fraction')
        ax.set_ylabel('Irrigated cropland cells (%)', fontsize=11)
        ax.set_xlabel('Date', fontsize=11)
        ax.set_ylim(0, 100)
        ax.legend(fontsize=9)

        # Mark growing season shading
        for yr in YEARS:
            gs_start = pd.Timestamp(yr, 4, 1)
            gs_end   = pd.Timestamp(yr, 9, 30)
            for ax in axes:
                ax.axvspan(gs_start, gs_end, alpha=0.07, color='green', zorder=0)

        plt.tight_layout()
        plt.savefig(figs / 'conus_human_et_timeseries.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved conus_human_et_timeseries.png')
    except Exception as e:
        print('HumanET timeseries skipped:', e)

Saved conus_human_et_timeseries.png


In [7]:
if True:
    try:
        # Compute 10-year mean growing-season HumanET per pixel
        gs_keys  = [(y, m) for y in YEARS for m in GROWING_SEASON if (y, m) in delta_maps]
        gs_stack = np.stack([delta_maps[k] for k in gs_keys], axis=0)
        gs_mean  = np.nanmean(gs_stack, axis=0)

        fig, ax = plt.subplots(figsize=(16, 7))
        vmax = np.nanpercentile(gs_mean[crop_mask_static], 95)
        im = ax.imshow(
            np.where(crop_mask_static, gs_mean, np.nan),
            extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
            origin='upper', cmap='Blues', vmin=0, vmax=max(vmax, 1), aspect='auto'
        )
        cbar = plt.colorbar(im, ax=ax, label='HumanET (mm/month)', fraction=0.025, pad=0.02)
        ax.set_title('Mean Growing-Season HumanET, CONUS Cropland (2015\u20132024 Apr\u2013Sep)',
                     fontsize=13)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')

        plt.tight_layout()
        plt.savefig(figs / 'conus_human_et_map_mean.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved conus_human_et_map_mean.png')
    except Exception as e:
        print('HumanET map skipped:', e)

/tmp/ipykernel_1657968/2675694889.py:6: RuntimeWarning: Mean of empty slice
  gs_mean  = np.nanmean(gs_stack, axis=0)


Saved conus_human_et_map_mean.png


## 4. SIF — OCO-2 Solar-Induced Fluorescence

Loads harmonized SIF half-monthly files and computes growing-season z-scores by pixel-month using a within-pixel climatology (2015–2024 mean/std per calendar month).

In [8]:
import re
from glob import glob

sif_files = sorted(sif_proc.glob('SIF_CONUS_*.nc'))
print('SIF files found:', len(sif_files))

# ---------------------------
# Pass 1: accumulate per-month climatology
# sif_clim[month] = list of 2D arrays
# ---------------------------
sif_clim  = {m: [] for m in range(1, 13)}
sif_meta  = []   # list of dicts: yyyymm, half, year, month, mean_sif

for fp in sif_files:
    stem = fp.stem   # e.g. SIF_CONUS_202207a
    m = re.search(r'(\d{6})([ab]?)$', stem)
    if m is None:
        print('  Skipping unrecognized filename:', fp.name)
        continue
    yyyymm = m.group(1)
    half   = m.group(2) if m.group(2) else 'a'
    year   = int(yyyymm[:4])
    month  = int(yyyymm[4:])

    if month not in GROWING_SEASON:
        continue

    try:
        ds = xr.open_dataset(fp)
        sif_var = None
        for vname in ['sif', 'SIF', 'sif_dc', 'fluorescence']:
            if vname in ds:
                sif_var = vname
                break
        if sif_var is None:
            sif_var = list(ds.data_vars)[0]
        arr = ds[sif_var].values
        ds.close()

        if arr.ndim == 3:
            arr = arr[0]
        arr = arr.astype(float)
        arr[arr < -900] = np.nan
        if arr.shape != (n_lat, n_lon):
            print(f'  SIF shape mismatch {fp.name}: {arr.shape}')
            continue

        sif_clim[month].append(arr)
        sif_meta.append({
            'yyyymm':   yyyymm,
            'half':     half,
            'year':     year,
            'month':    month,
            'mean_sif': float(np.nanmean(arr[crop_mask_static]))
        })
    except Exception as e:
        print(f'  SIF load error {fp.name}: {e}')

# ---------------------------
# Compute pixel-wise climatology per calendar month
# ---------------------------
sif_clim_mean = {}
sif_clim_std  = {}
for mo, arrs in sif_clim.items():
    if len(arrs) > 0:
        stack = np.stack(arrs, axis=0)
        sif_clim_mean[mo] = np.nanmean(stack, axis=0)
        sif_clim_std[mo]  = np.nanstd(stack,  axis=0)

# ---------------------------
# Pass 2: compute z-scores; store aggregate cropland stats
# ---------------------------
sif_zscore_maps = {}   # (yyyymm, half) -> 2D z-score array

for fp in sif_files:
    stem = fp.stem
    m = re.search(r'(\d{6})([ab]?)$', stem)
    if m is None:
        continue
    yyyymm = m.group(1)
    half   = m.group(2) if m.group(2) else 'a'
    year   = int(yyyymm[:4])
    month  = int(yyyymm[4:])

    if month not in GROWING_SEASON:
        continue
    if month not in sif_clim_mean:
        continue

    try:
        ds = xr.open_dataset(fp)
        sif_var = None
        for vname in ['sif', 'SIF', 'sif_dc', 'fluorescence']:
            if vname in ds:
                sif_var = vname
                break
        if sif_var is None:
            sif_var = list(ds.data_vars)[0]
        arr = ds[sif_var].values
        ds.close()
        if arr.ndim == 3:
            arr = arr[0]
        arr = arr.astype(float)
        arr[arr < -900] = np.nan
        if arr.shape != (n_lat, n_lon):
            continue

        mu  = sif_clim_mean[month]
        sig = sif_clim_std[month]
        with np.errstate(invalid='ignore', divide='ignore'):
            z = np.where(sig > 0, (arr - mu) / sig, np.nan)

        sif_zscore_maps[(yyyymm, half)] = z
    except Exception as e:
        print(f'  SIF z-score error {fp.name}: {e}')

df_sif = pd.DataFrame(sif_meta)
if len(df_sif) > 0:
    df_sif['date'] = pd.to_datetime(df_sif['yyyymm'], format='%Y%m')
    print('SIF records (growing season):', len(df_sif))
    print('Mean SIF by month:')
    print(df_sif.groupby('month')['mean_sif'].mean().round(4).to_string())
    print()
    print('Z-score maps computed:', len(sif_zscore_maps))
else:
    print('No SIF files loaded.')

SIF files found: 235


/tmp/ipykernel_1657968/2458441096.py:67: RuntimeWarning: Mean of empty slice
  sif_clim_mean[mo] = np.nanmean(stack, axis=0)
/home/pielab-sandbox-jcoldiron/SIF-Analysis/.venv/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


SIF records (growing season): 117
Mean SIF by month:
month
4    0.0591
5    0.1117
6    0.2166
7    0.3308
8    0.2917
9    0.1561

Z-score maps computed: 117


## 5. Drought Categorization

Loads GRIDMET drought indices and USDM categories. Bins SPEI-90d (the 90-day SPEI) into USDM-equivalent DM categories for comparison with SIF anomalies.

In [9]:
gmet_files = sorted(gmet_proc.glob('GRIDMET_drought_*.tif'))
print('GRIDMET drought files found:', len(gmet_files))

drought_maps = {}   # (year, month) -> dict of band arrays
drought_recs = []

for fp in gmet_files:
    m = re.search(r'(\d{6})', fp.stem)
    if m is None:
        continue
    yyyymm = m.group(1)
    year   = int(yyyymm[:4])
    month  = int(yyyymm[4:])

    try:
        with rasterio.open(fp) as src:
            n_bands = src.count
            tags    = src.tags()
            bands   = {}
            for b in range(1, n_bands + 1):
                band_tags = src.tags(b)
                bname = band_tags.get('name', band_tags.get('STATISTICS_MINIMUM', f'band{b}'))
                arr = src.read(b).astype(float)
                nd  = src.nodata
                if nd is not None:
                    arr[arr == nd] = np.nan
                bands[f'band{b}'] = arr
                if 'spei90' in bname.lower() or 'spei_90' in bname.lower() or b == 1:
                    bands['spei90d'] = arr
            if 'spei90d' not in bands:
                bands['spei90d'] = bands['band1']

        spei90 = bands['spei90d']
        if spei90.shape == (n_lat, n_lon):
            dm = spei_to_dm(spei90)
            dm_masked = np.where(crop_mask_static, dm, np.nan)
            drought_maps[(year, month)] = {
                'spei90d': spei90,
                'dm_cat':  dm,
                'dm_cat_crop': dm_masked
            }
            drought_recs.append({
                'year':          year,
                'month':         month,
                'yyyymm':        yyyymm,
                'mean_spei90d':  float(np.nanmean(spei90[crop_mask_static])),
                'mean_dm_cat':   float(np.nanmean(dm_masked)),
                'frac_d2plus':   float(np.nanmean(dm_masked >= 2))
            })
    except Exception as e:
        print(f'  GRIDMET error {fp.name}: {e}')

# Load USDM for comparison
usdm_recs = []
usdm_maps = {}
usdm_files = sorted(usdm_proc.glob('USDM_CONUS_*.tif'))
print('USDM files found:', len(usdm_files))

for fp in usdm_files:
    m = re.search(r'(\d{6})', fp.stem)
    if m is None:
        continue
    yyyymm = m.group(1)
    year   = int(yyyymm[:4])
    month  = int(yyyymm[4:])
    try:
        with rasterio.open(fp) as src:
            arr = src.read(1).astype(float)
            nd  = src.nodata
            if nd is not None:
                arr[arr == nd] = np.nan
        if arr.shape == (n_lat, n_lon):
            dm_crop = np.where(crop_mask_static, arr, np.nan)
            usdm_maps[(year, month)] = arr
            usdm_recs.append({
                'year': year, 'month': month, 'yyyymm': yyyymm,
                'mean_usdm': float(np.nanmean(dm_crop))
            })
    except Exception as e:
        print(f'  USDM error {fp.name}: {e}')

df_drought = pd.DataFrame(drought_recs)
df_usdm    = pd.DataFrame(usdm_recs)

if len(df_drought) > 0 and len(df_usdm) > 0:
    merged = pd.merge(df_drought, df_usdm, on=['year', 'month', 'yyyymm'], how='inner')
    r, p = stats.pearsonr(merged['mean_dm_cat'].dropna(), merged['mean_usdm'].dropna())
    print('Pearson r (SPEI-DM vs USDM): ' + str(round(r, 3)) + '  p=' + str(round(p, 4)))
else:
    print('Insufficient data for SPEI vs USDM comparison.')

print('GRIDMET drought records:', len(drought_recs))
print('USDM records:           ', len(usdm_recs))

GRIDMET drought files found: 120
USDM files found: 240
Insufficient data for SPEI vs USDM comparison.
GRIDMET drought records: 120
USDM records:            0


In [10]:
if True:
    try:
        fig, axes = plt.subplots(2, 2, figsize=(18, 10))
        months_plot = [(2022, 7), (2022, 8)]

        dm_cmap  = mcolors.ListedColormap(['#A8D5A2', '#FFFF00', '#F5BE4E', '#E08C32', '#C74B2A', '#6B0F0F'])
        dm_norm  = mcolors.BoundaryNorm([-1.5, -0.5, 0.5, 1.5, 2.5, 3.5, 4.5], dm_cmap.N)
        dm_ticks = [-1, 0, 1, 2, 3, 4]
        dm_tlabs = ['None', 'D0', 'D1', 'D2', 'D3', 'D4']

        for col, (yr, mo) in enumerate(months_plot):
            mo_label = date(yr, mo, 1).strftime('%B %Y')

            # SPEI-derived DM (top row)
            ax = axes[0][col]
            if (yr, mo) in drought_maps:
                dm_arr = drought_maps[(yr, mo)]['dm_cat']
                im = ax.imshow(
                    dm_arr,
                    extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
                    origin='upper', cmap=dm_cmap, norm=dm_norm, aspect='auto'
                )
                cbar = plt.colorbar(im, ax=ax, ticks=dm_ticks)
                cbar.ax.set_yticklabels(dm_tlabs)
            ax.set_title(f'SPEI-90d \u2192 DM: {mo_label}', fontsize=11)
            ax.set_xlabel('Longitude')
            ax.set_ylabel('Latitude')

            # USDM (bottom row)
            ax = axes[1][col]
            if (yr, mo) in usdm_maps:
                im = ax.imshow(
                    usdm_maps[(yr, mo)],
                    extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
                    origin='upper', cmap=dm_cmap, norm=dm_norm, aspect='auto'
                )
                cbar = plt.colorbar(im, ax=ax, ticks=dm_ticks)
                cbar.ax.set_yticklabels(dm_tlabs)
            ax.set_title(f'USDM: {mo_label}', fontsize=11)
            ax.set_xlabel('Longitude')
            ax.set_ylabel('Latitude')

        plt.suptitle('Drought Comparison: SPEI-90d-derived DM vs USDM, Summer 2022',
                     fontsize=13, y=1.01)
        plt.tight_layout()
        plt.savefig(figs / 'conus_drought_comparison_2022.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved conus_drought_comparison_2022.png')
    except Exception as e:
        print('Drought comparison map skipped:', e)

Saved conus_drought_comparison_2022.png


## 5b. Data Quality Checks

Per-year raster maps (July reference month) to verify that each input dataset loaded correctly and covers CONUS. Run each cell independently to isolate problems.

| Cell | Variable | What to look for |
|------|----------|-----------------|
| CDL | Cropland fraction | Consistent corn-belt pattern each year; no blank years |
| HumanET | OpenET − NLDAS | Positive values in irrigated regions (High Plains, CA, PNW); zeros elsewhere |
| SIF z-score | OCO-2 SIF anomaly | Spatially coherent signal; no year entirely NaN |
| GRIDMET SPEI-90d | Drought index | Plausible drought patterns (red = dry); 2012, 2022 should look severe |
| USDM | Drought Monitor | Matches SPEI patterns; cross-validates GRIDMET |

**Other QC to watch for:**
- Blank or all-NaN maps → file missing or wrong path
- Uniform constant values → nodata fill not masked
- Shape mismatch warnings printed above → regridding needed
- Irrigated fraction (Irr%) near 0% in HumanET table → ΔET signal lost, check NLDAS units

In [11]:
QC_MONTH = 7  # July — peak growing season reference month for all QC plots

# QC — CDL cropland fraction per year
print('=== CDL Cropland Fraction — per year ===')
print('Year    Valid%     Mean frac   Mask cells   Mask%')
for yi, year in enumerate(YEARS):
    arr = cropland_frac_all[yi]
    valid = np.isfinite(arr)
    mask  = arr >= CDL_CROP_FRAC_MIN
    pct_mask = 100.0 * mask.sum() / arr.size
    mean_f   = float(np.nanmean(arr)) if valid.any() else float('nan')
    print(str(year) + '  ' + f'{100*valid.mean():.1f}%   {mean_f:.3f}   {mask.sum():>8,}   {pct_mask:.2f}%')

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for yi, (year, ax) in enumerate(zip(YEARS, axes.flatten())):
    arr = cropland_frac_all[yi]
    if not np.any(np.isfinite(arr)):
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(str(year)); ax.axis('off'); continue
    im = ax.imshow(np.where(np.isfinite(arr), arr, np.nan),
                   extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
                   origin='upper', cmap='YlGn', vmin=0, vmax=1, aspect='auto')
    ax.contour(CONUS_LON, CONUS_LAT, (arr >= CDL_CROP_FRAC_MIN).astype(float),
               levels=[0.5], colors='k', linewidths=0.5, alpha=0.6)
    ax.set_title(str(year), fontsize=11); ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('QC: CDL Cropland Fraction — per year, 2015–2024\n(Black contour = ≥50% cropland mask threshold)', fontsize=13)
plt.tight_layout()
plt.savefig(figs / 'qc_cdl_fraction_peryear.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: qc_cdl_fraction_peryear.png')

=== CDL Cropland Fraction — per year ===
Year    Valid%     Mean frac   Mask cells   Mask%
2015  100.0%   0.418     25,087   40.84%
2016  100.0%   0.419     25,193   41.01%
2017  100.0%   0.420     25,216   41.05%
2018  100.0%   0.422     25,407   41.36%
2019  100.0%   0.422     25,386   41.33%
2020  100.0%   0.422     25,431   41.40%
2021  100.0%   0.422     25,443   41.42%
2022  100.0%   0.421     25,388   41.33%
2023  100.0%   0.421     25,389   41.33%
2024  100.0%   0.403     24,317   39.59%
Saved: qc_cdl_fraction_peryear.png


In [12]:
# QC — HumanET (deltaET) per year (July)
print('=== HumanET (ΔET) — July per year (all pixels) ===')
print('Year    Valid%     Mean     Min    Max    Irr%')
for year in YEARS:
    arr = delta_maps.get((year, QC_MONTH))
    if arr is None:
        print(str(year) + '  MISSING')
        continue
    valid = np.isfinite(arr)
    irr_pct = 100.0 * (arr[valid] > IRR_THRESHOLD_MM).mean() if valid.any() else float('nan')
    print(str(year) + '  ' + f'{100*valid.mean():.1f}%   {np.nanmean(arr):.1f}   {np.nanmin(arr):.1f}   {np.nanmax(arr):.1f}   {irr_pct:.1f}%')

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for yi, (year, ax) in enumerate(zip(YEARS, axes.flatten())):
    arr = delta_maps.get((year, QC_MONTH))
    if arr is None:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(str(year)); ax.axis('off'); continue
    display = np.where(np.isfinite(arr), arr, np.nan)
    im = ax.imshow(display,
                   extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
                   origin='upper', cmap='Blues', vmin=0, vmax=100, aspect='auto')
    ax.set_title(str(year), fontsize=11); ax.set_xticks([]); ax.set_yticks([])
    # Mark irrigation threshold contour
    ax.contour(CONUS_LON, CONUS_LAT, display, levels=[IRR_THRESHOLD_MM],
               colors='red', linewidths=0.5, alpha=0.5)

fig.suptitle('QC: HumanET (ΔET = OpenET − NLDAS) — July, 2015–2024\n'
             '(mm/mo, all valid pixels; red contour = irrigation threshold)', fontsize=12)
plt.tight_layout()
plt.savefig(figs / 'qc_humanet_july_peryear.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: qc_humanet_july_peryear.png')


=== HumanET (ΔET) — July per year (all pixels) ===
Year    Valid%     Mean     Min    Max    Irr%
2015  12.3%   26.7   -116.6   194.1   53.2%
2016  12.4%   24.8   -79.7   183.3   46.8%
2017  12.5%   31.7   -91.9   181.8   62.5%
2018  12.8%   32.3   -63.6   204.7   61.9%
2019  12.7%   24.9   -62.1   169.4   45.7%
2020  12.7%   30.1   -53.9   198.4   57.5%
2021  12.8%   34.4   -64.3   214.7   64.3%
2022  12.7%   32.6   -70.1   193.9   63.6%
2023  12.7%   42.0   -133.4   233.8   81.4%
2024  11.1%   26.7   -78.5   219.4   46.9%
Saved: qc_humanet_july_peryear.png


In [13]:
# QC — SIF z-score per year (July, both halves merged)
#
# Key facts about this dataset:
#   - Spatial extent: lon -124.7 to -84.2 W (eastern US east of ~Ohio/Georgia NOT covered)
#   - OCO-2 orbital gaps cause ~21% NaN even within coverage area
#   - Merging both half-months (a+b) reduces gaps slightly
#   - This QC plot shows ALL pixels (not cropland-only) so you can diagnose coverage
#   - Values below ~26.5N may include a few pixels of northern Mexico

print('=== SIF z-score — July per year (all pixels, a+b merged) ===')
print('Year    Valid%     Mean z   Min    Max    Note')
for year in YEARS:
    # Merge both half-months to maximise coverage
    halves = []
    for half in ['a', 'b', '']:
        key = (f'{year}07', half)
        if key in sif_zscore_maps:
            halves.append(sif_zscore_maps[key])
    if not halves:
        print(str(year) + '  MISSING')
        continue
    arr = np.nanmean(np.stack(halves, axis=0), axis=0) if len(halves) > 1 else halves[0]
    valid = np.isfinite(arr)
    note = '(east of -84W not in dataset)'
    print(str(year) + '  ' + f'{100*valid.mean():.1f}%   {np.nanmean(arr):.3f}   '
          + f'{np.nanmin(arr):.2f}   {np.nanmax(arr):.2f}   {note}')

# Plot — one panel per year, all pixels shown (no crop mask)
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for yi, (year, ax) in enumerate(zip(YEARS, axes.flatten())):
    halves = []
    for half in ['a', 'b', '']:
        key = (f'{year}07', half)
        if key in sif_zscore_maps:
            halves.append(sif_zscore_maps[key])
    if not halves:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(str(year)); ax.axis('off'); continue
    arr = np.nanmean(np.stack(halves, axis=0), axis=0) if len(halves) > 1 else halves[0]
    pct = 100 * np.isfinite(arr).mean()
    im = ax.imshow(
        arr,
        extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
        origin='upper', cmap='RdYlGn', vmin=-2, vmax=2, aspect='auto'
    )
    ax.set_title(f'{year}  ({pct:.0f}% valid)', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    # Mark eastern data cutoff at -84.2W
    ax.axvline(-84.1875, color='blue', linewidth=0.8, linestyle='--', alpha=0.7)
    # Mark approximate Mexico boundary
    ax.axhline(26.5, color='orange', linewidth=0.8, linestyle=':', alpha=0.7)

# Shared colorbar
cbar = fig.colorbar(
    plt.cm.ScalarMappable(cmap='RdYlGn',
                          norm=plt.Normalize(vmin=-2, vmax=2)),
    ax=axes, orientation='vertical', fraction=0.01, pad=0.01
)
cbar.set_label('SIF z-score', fontsize=11)
fig.suptitle(
    'QC: SIF Z-score (all pixels) — July, 2015\u20132024\n'
    'Green = above-normal SIF | Red = below-normal | White = no OCO-2 data\n'
    'Blue dashed = eastern data boundary (-84.2\u00b0W) | Orange dotted = ~Mexico border (26.5\u00b0N)',
    fontsize=11
)
plt.savefig(figs / 'qc_sif_zscore_july_peryear.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: qc_sif_zscore_july_peryear.png')
print()
print('NOTE: Data extends only to -84.2W — eastern US (east of Ohio/Georgia) is not in this dataset.')
print('NOTE: ~21% of pixels are NaN in any given half-month due to OCO-2 orbital sampling gaps.')
print('NOTE: A few pixels at lat < 26.5N may include northern Mexico (orange line).')


=== SIF z-score — July per year (all pixels, a+b merged) ===
Year    Valid%     Mean z   Min    Max    Note
2015  78.8%   0.058   -2.64   2.82   (east of -84W not in dataset)
2016  78.9%   0.257   -2.41   2.41   (east of -84W not in dataset)
2017  78.9%   -0.138   -2.84   2.33   (east of -84W not in dataset)
2018  78.9%   0.033   -2.63   2.96   (east of -84W not in dataset)
2019  78.9%   0.389   -2.68   2.62   (east of -84W not in dataset)
2020  78.9%   0.060   -2.31   2.48   (east of -84W not in dataset)
2021  78.8%   -0.164   -2.77   2.80   (east of -84W not in dataset)
2022  78.8%   -0.259   -2.73   2.51   (east of -84W not in dataset)
2023  78.9%   0.005   -2.63   2.77   (east of -84W not in dataset)
2024  78.9%   -0.242   -2.89   2.63   (east of -84W not in dataset)


/tmp/ipykernel_1657968/2304495638.py:22: RuntimeWarning: Mean of empty slice
  arr = np.nanmean(np.stack(halves, axis=0), axis=0) if len(halves) > 1 else halves[0]
/tmp/ipykernel_1657968/2304495638.py:39: RuntimeWarning: Mean of empty slice
  arr = np.nanmean(np.stack(halves, axis=0), axis=0) if len(halves) > 1 else halves[0]


Saved: qc_sif_zscore_july_peryear.png

NOTE: Data extends only to -84.2W — eastern US (east of Ohio/Georgia) is not in this dataset.
NOTE: ~21% of pixels are NaN in any given half-month due to OCO-2 orbital sampling gaps.
NOTE: A few pixels at lat < 26.5N may include northern Mexico (orange line).


In [14]:
# QC — GRIDMET SPEI-90d per year (July)
print('=== GRIDMET SPEI-90d — July per year ===')
print('Year    Valid%     Mean SPEI   Min    Max')
for year in YEARS:
    entry = drought_maps.get((year, QC_MONTH))
    if entry is None:
        print(str(year) + '  MISSING')
        continue
    arr   = entry['spei90d']
    valid = np.isfinite(arr)
    print(str(year) + '  ' + f'{100*valid.mean():.1f}%   {np.nanmean(arr):.3f}   {np.nanmin(arr):.2f}   {np.nanmax(arr):.2f}')

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for yi, (year, ax) in enumerate(zip(YEARS, axes.flatten())):
    entry = drought_maps.get((year, QC_MONTH))
    arr   = entry['spei90d'] if entry is not None else None
    if arr is None or not np.any(np.isfinite(arr)):
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(str(year)); ax.axis('off'); continue
    im = ax.imshow(np.where(np.isfinite(arr), arr, np.nan),
                   extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
                   origin='upper', cmap='RdBu', vmin=-3, vmax=3, aspect='auto')
    ax.set_title(str(year), fontsize=11); ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('QC: GRIDMET SPEI-90d — July, 2015–2024\n(Red = drought, Blue = wet)', fontsize=13)
plt.tight_layout()
plt.savefig(figs / 'qc_gridmet_spei_july_peryear.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: qc_gridmet_spei_july_peryear.png')

=== GRIDMET SPEI-90d — July per year ===
Year    Valid%     Mean SPEI   Min    Max
2015  100.0%   0.297   -1.98   2.09
2016  100.0%   -0.024   -2.07   2.09
2017  100.0%   -0.161   -2.09   2.09
2018  100.0%   0.057   -2.09   2.09
2019  100.0%   0.075   -2.09   2.09
2020  100.0%   0.025   -2.01   2.09
2021  100.0%   0.064   -2.09   2.09
2022  100.0%   -0.119   -2.09   2.09
2023  100.0%   -0.115   -2.09   2.09
2024  100.0%   0.012   -2.09   2.09
Saved: qc_gridmet_spei_july_peryear.png


In [15]:
# QC — USDM drought per year (July reference month)

dm_cmap = mcolors.ListedColormap(['#A8D5A2','#FFFF00','#F5BE4E','#E08C32','#C74B2A','#6B0F0F'])
dm_norm = mcolors.BoundaryNorm([-1.5,-0.5,0.5,1.5,2.5,3.5,4.5], dm_cmap.N)

print('=== USDM — July per year ===')
print('Year    Valid%     Mean DM   D2+ frac')
for year in YEARS:
    arr = usdm_maps.get((year, QC_MONTH))
    if arr is None:
        print(str(year) + '  MISSING')
        continue
    valid = np.isfinite(arr)
    d2p = 100.0 * (arr[valid] >= 2).mean() if valid.any() else float('nan')
    print(str(year) + '  ' + f'{100*valid.mean():.1f}%   {np.nanmean(arr):.2f}   {d2p:.1f}%')

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for yi, (year, ax) in enumerate(zip(YEARS, axes.flatten())):
    arr = usdm_maps.get((year, QC_MONTH))
    if arr is None or not np.any(np.isfinite(arr)):
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(str(year)); ax.axis('off'); continue
    ax.imshow(np.where(np.isfinite(arr), arr, np.nan),
              extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
              origin='upper', cmap=dm_cmap, norm=dm_norm, aspect='auto')
    ax.set_title(str(year), fontsize=11); ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('QC: USDM Drought Monitor — July, 2015–2024', fontsize=13)
plt.tight_layout()
plt.savefig(figs / 'qc_usdm_july_peryear.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: qc_usdm_july_peryear.png')

=== USDM — July per year ===
Year    Valid%     Mean DM   D2+ frac
2015  MISSING
2016  MISSING
2017  MISSING
2018  MISSING
2019  MISSING
2020  MISSING
2021  MISSING
2022  MISSING
2023  MISSING
2024  MISSING


Saved: qc_usdm_july_peryear.png


## 6. SIF × Drought — CONUS Cropland

The core analysis: does SIF anomaly track drought severity? Do irrigated crops maintain higher SIF under drought compared to rainfed crops?

This section reproduces Figures 7–12 from the Iowa analysis for all CONUS cropland.

In [16]:
# Build combined pixel-month DataFrame
# Join SIF z-scores, HumanET (irrigation flag), and drought index for every
# cropland pixel across all growing-season months.
#
# With ~25K cropland cells × 60 month-year combos = ~1.5M rows, no sampling needed.

crop_idx = np.argwhere(crop_mask_static)   # (n_cells, 2) — row/col indices of cropland cells

rows_s = crop_idx[:, 0]
cols_s = crop_idx[:, 1]
lats_s = CONUS_LAT[rows_s]
lons_s = CONUS_LON[cols_s]

n_crop = len(crop_idx)
print(f'Cropland pixels: {n_crop:,}')
print(f'Expected rows (all GS months): {n_crop * len(YEARS) * len(GROWING_SEASON):,}')

combined_rows = []

for year in YEARS:
    for month in GROWING_SEASON:
        yyyymm = f'{year}{month:02d}'

        # HumanET delta — skip month if unavailable
        if (year, month) not in delta_maps:
            continue
        delta_arr = delta_maps[(year, month)]

        # Drought indices
        if (year, month) in drought_maps:
            dm_arr   = drought_maps[(year, month)]['dm_cat']
            spei_arr = drought_maps[(year, month)]['spei90d']
        else:
            dm_arr   = np.full((n_lat, n_lon), np.nan)
            spei_arr = np.full((n_lat, n_lon), np.nan)

        # SIF z-score: prefer first-half ('a'), fall back to 'b' then no suffix
        sif_z_arr = None
        for half in ['a', 'b', '']:
            if (yyyymm, half) in sif_zscore_maps:
                sif_z_arr = sif_zscore_maps[(yyyymm, half)]
                break

        # Extract all cropland pixels at once via array indexing (vectorized)
        delta_s  = delta_arr[rows_s, cols_s]
        dm_s     = dm_arr[rows_s, cols_s]
        spei_s   = spei_arr[rows_s, cols_s]
        sif_z_s  = sif_z_arr[rows_s, cols_s] if sif_z_arr is not None else np.full(n_crop, np.nan)
        is_irr_s = np.where(np.isnan(delta_s), np.nan, (delta_s > IRR_THRESHOLD_MM).astype(float))

        # Build one record per pixel — pandas concat is faster than per-pixel dict appends
        month_df = pd.DataFrame({
            'year':         year,
            'month':        month,
            'yyyymm':       yyyymm,
            'lat':          lats_s,
            'lon':          lons_s,
            'sif_z':        sif_z_s,
            'delta_et':     delta_s,
            'is_irrigated': is_irr_s,
            'dm_cat':       dm_s,
            'spei90d':      spei_s,
        })
        combined_rows.append(month_df)

df_combined = pd.concat(combined_rows, ignore_index=True)
df_combined['date'] = pd.to_datetime(df_combined['yyyymm'], format='%Y%m')

# Drop rows where both sif_z and dm_cat are NaN (no useful data)
df_combined = df_combined.dropna(subset=['sif_z', 'dm_cat'], how='all')
df_combined['dm_cat_int'] = df_combined['dm_cat'].round().astype('Int64')

print('Combined pixel-month records:', f'{len(df_combined):,}')
print('Columns:', list(df_combined.columns))
print()
if len(df_combined) > 0:
    print('DM category distribution:')
    for c in sorted(df_combined['dm_cat_int'].dropna().unique()):
        n = (df_combined['dm_cat_int'] == c).sum()
        label = DM_LABELS.get(int(c), str(c))
        print('  ' + str(c) + ' (' + label + '): ' + f'{n:,}')


Cropland pixels: 25,153
Expected rows (all GS months): 1,509,180
Combined pixel-month records: 1,509,180
Columns: ['year', 'month', 'yyyymm', 'lat', 'lon', 'sif_z', 'delta_et', 'is_irrigated', 'dm_cat', 'spei90d', 'date', 'dm_cat_int']

DM category distribution:
  -1 (No drought): 1,363,529
  0 (D0): 81,465
  1 (D1): 46,089
  2 (D2): 16,532
  3 (D3): 1,565


In [17]:
# Save df_combined and the cropland mask for use in the regression notebook (02_...).
# This allows notebook 02 to run independently without reloading all source data.

_out = proc / 'regression'
_out.mkdir(parents=True, exist_ok=True)

_parquet_path = _out / 'df_combined_gs.parquet'
_mask_path    = _out / 'crop_mask_static.npy'

if len(df_combined) > 0:
    # Pandas nullable integer types (Int64) conflict with pyarrow — cast to float first.
    _df_save = df_combined.copy()
    for col in _df_save.columns:
        if hasattr(_df_save[col], 'dtype') and str(_df_save[col].dtype) in ('Int8','Int16','Int32','Int64'):
            _df_save[col] = _df_save[col].astype('float64')
    # Drop the date column (datetime reconstructed on load from yyyymm)
    _df_save = _df_save.drop(columns=['date'], errors='ignore')
    _df_save.to_parquet(_parquet_path, index=False)
    np.save(_mask_path, crop_mask_static)
    print('Saved df_combined to: ' + str(_parquet_path))
    print('Saved crop_mask_static to: ' + str(_mask_path))
    print('Shape:', _df_save.shape, '  Columns:', list(_df_save.columns))
else:
    print('df_combined is empty — nothing saved.')


Saved df_combined to: /home/pielab-sandbox-jcoldiron/SIF-Analysis/data/processed/conus/regression/df_combined_gs.parquet
Saved crop_mask_static to: /home/pielab-sandbox-jcoldiron/SIF-Analysis/data/processed/conus/regression/crop_mask_static.npy
Shape: (1509180, 11)   Columns: ['year', 'month', 'yyyymm', 'lat', 'lon', 'sif_z', 'delta_et', 'is_irrigated', 'dm_cat', 'spei90d', 'dm_cat_int']


In [18]:
if True:
    try:
        # -- Build monthly CONUS-mean SIF z-score time series --
        if len(df_sif) > 0:
            # Aggregate z-scores from maps
            ts_rows = []
            for (yyyymm, half), z_map in sif_zscore_maps.items():
                mo = int(yyyymm[4:])
                yr = int(yyyymm[:4])
                z_crop = z_map[crop_mask_static]
                ts_rows.append({
                    'year': yr, 'month': mo, 'yyyymm': yyyymm, 'half': half,
                    'mean_sif_z': float(np.nanmean(z_crop))
                })
            df_ts = pd.DataFrame(ts_rows)
            df_ts['date'] = pd.to_datetime(df_ts['yyyymm'], format='%Y%m')
            df_ts = df_ts.sort_values('date')

            # USDM area fractions per month
            usdm_frac_recs = []
            for (yr, mo), usdm_arr in usdm_maps.items():
                u_crop = usdm_arr[crop_mask_static]
                valid  = ~np.isnan(u_crop)
                total  = valid.sum()
                row = {'year': yr, 'month': mo}
                for cat in [0, 1, 2, 3, 4]:
                    row[f'frac_d{cat}'] = (u_crop[valid] >= cat).sum() / total if total > 0 else np.nan
                usdm_frac_recs.append(row)
            df_usdm_frac = pd.DataFrame(usdm_frac_recs)
            df_usdm_frac['date'] = pd.to_datetime(
                df_usdm_frac['year'].astype(str) + df_usdm_frac['month'].astype(str).str.zfill(2),
                format='%Y%m'
            )

            # --- Plot ---
            fig, axes = plt.subplots(3, 1, figsize=(18, 11), sharex=True)

            # Panel 1: SIF z-score time series
            ax = axes[0]
            ax.plot(df_ts['date'], df_ts['mean_sif_z'], color='#2a7b0f', linewidth=1.5, label='CONUS cropland mean SIF z-score')
            ax.axhline(0, color='k', linewidth=0.7, linestyle='--')
            ax.fill_between(df_ts['date'], df_ts['mean_sif_z'], 0,
                            where=df_ts['mean_sif_z'] < 0, alpha=0.3, color='red', label='Negative anomaly')
            ax.fill_between(df_ts['date'], df_ts['mean_sif_z'], 0,
                            where=df_ts['mean_sif_z'] > 0, alpha=0.3, color='green', label='Positive anomaly')
            ax.set_ylabel('SIF z-score', fontsize=11)
            ax.set_title('CONUS Cropland SIF z-score and Drought Severity, 2015\u20132024', fontsize=13)
            ax.legend(fontsize=9, loc='upper left')

            # Panel 2: USDM area fraction stacked
            ax = axes[1]
            if len(df_usdm_frac) > 0:
                df_usdm_frac = df_usdm_frac.sort_values('date')
                cats_plot = [(4, '#6B0F0F', 'D4'), (3, '#C74B2A', 'D3'),
                             (2, '#E08C32', 'D2'), (1, '#F5BE4E', 'D1'), (0, '#FFFF00', 'D0')]
                prev = np.zeros(len(df_usdm_frac))
                for cat, color, label in cats_plot:
                    col_name = f'frac_d{cat}'
                    if col_name in df_usdm_frac.columns:
                        vals = df_usdm_frac[col_name].fillna(0).values
                        ax.bar(df_usdm_frac['date'], vals, bottom=prev,
                               color=color, width=25, label=label, alpha=0.85)
                        prev = prev + vals
            ax.set_ylabel('USDM drought area fraction', fontsize=11)
            ax.set_ylim(0, 1)
            ax.legend(fontsize=9, loc='upper left', ncol=5)

            # Panel 3: SPEI90d CONUS-mean
            ax = axes[2]
            if len(df_drought) > 0:
                df_drought['date'] = pd.to_datetime(df_drought['yyyymm'], format='%Y%m')
                df_d_gs = df_drought[df_drought['month'].isin(GROWING_SEASON)].sort_values('date')
                ax.bar(df_d_gs['date'],
                       df_d_gs['mean_spei90d'],
                       color=df_d_gs['mean_spei90d'].apply(lambda v: '#C74B2A' if v < -1.0 else ('#F5BE4E' if v < 0 else '#A8D5A2')),
                       width=25, alpha=0.8)
                ax.axhline(0,    color='k',   linewidth=0.7)
                ax.axhline(-1.0, color='red', linewidth=0.8, linestyle='--', alpha=0.6, label='D1 threshold')
                ax.axhline(-1.5, color='darkred', linewidth=0.8, linestyle='--', alpha=0.6, label='D2 threshold')
                ax.set_ylabel('Cropland mean SPEI-90d', fontsize=11)
                ax.legend(fontsize=9)
            ax.set_xlabel('Date', fontsize=11)

            plt.tight_layout()
            plt.savefig(figs / 'conus_sif_zscore_timeseries.png', dpi=150, bbox_inches='tight')
            plt.show()
            print('Saved conus_sif_zscore_timeseries.png')
        else:
            print('No SIF data available for timeseries plot.')
    except Exception as e:
        print('SIF timeseries skipped:', e)

SIF timeseries skipped: 'year'


In [19]:
if True:
    try:
        if len(df_combined) == 0 or df_combined['sif_z'].isna().all():
            print('No combined data for violin plot.')
        else:
            import warnings

            df_plot = df_combined.dropna(subset=['sif_z', 'dm_cat_int', 'is_irrigated'])
            present_cats = sorted(df_plot['dm_cat_int'].dropna().unique())
            cat_labels   = [DM_LABELS.get(int(c), str(c)) for c in present_cats]
            cat_colors   = [DM_COLORS.get(int(c), '#aaaaaa') for c in present_cats]

            fig, axes = plt.subplots(1, 2, figsize=(16, 7))
            titles = ['Irrigated (HumanET > 20 mm/mo)', 'Rainfed (HumanET \u2264 20 mm/mo)']
            irr_flags = [1, 0]

            for ax, title, irr_flag in zip(axes, titles, irr_flags):
                sub = df_plot[df_plot['is_irrigated'] == irr_flag]
                if len(sub) == 0:
                    ax.set_title(title + ' (no data)')
                    continue

                groups = [sub[sub['dm_cat_int'] == c]['sif_z'].dropna().values for c in present_cats]
                groups = [g for g in groups if len(g) > 1]

                if len(groups) == 0:
                    ax.set_title(title + ' (insufficient data)')
                    continue

                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    parts = ax.violinplot(groups, positions=range(len(groups)),
                                         showmedians=True, showextrema=False)

                for pi, (pc, color) in enumerate(zip(parts['bodies'], cat_colors)):
                    pc.set_facecolor(color)
                    pc.set_alpha(0.75)
                parts['cmedians'].set_color('black')
                parts['cmedians'].set_linewidth(2)

                ax.axhline(0, color='k', linewidth=0.8, linestyle='--', alpha=0.5)
                ax.set_xticks(range(len(groups)))
                ax.set_xticklabels(cat_labels[:len(groups)], fontsize=10)
                ax.set_xlabel('Drought category', fontsize=11)
                ax.set_ylabel('SIF z-score', fontsize=11)
                ax.set_title(title, fontsize=12)

                # Add n labels
                for pi, cat in enumerate(present_cats[:len(groups)]):
                    n = (sub['dm_cat_int'] == cat).sum()
                    ax.text(pi, ax.get_ylim()[0] + 0.05, f'n={n:,}',
                            ha='center', fontsize=7, color='gray')

            plt.suptitle('CONUS Cropland SIF z-score by Drought Category', fontsize=13)
            plt.tight_layout()
            plt.savefig(figs / 'conus_sif_violin_by_drought.png', dpi=150, bbox_inches='tight')
            plt.show()
            print('Saved conus_sif_violin_by_drought.png')
    except Exception as e:
        print('Violin plot skipped:', e)

Saved conus_sif_violin_by_drought.png


In [20]:
if True:
    try:
        if len(df_combined) == 0:
            print('No combined data for ET-by-drought plot.')
        else:
            df_plot = df_combined.dropna(subset=['delta_et', 'dm_cat_int', 'is_irrigated'])
            present_cats = sorted(df_plot['dm_cat_int'].dropna().unique())
            cat_labels   = [DM_LABELS.get(int(c), str(c)) for c in present_cats]

            fig, ax = plt.subplots(figsize=(12, 6))

            x   = np.arange(len(present_cats))
            w   = 0.35

            irr_means = [df_plot[(df_plot['dm_cat_int'] == c) & (df_plot['is_irrigated'] == 1)]['delta_et'].mean()
                         for c in present_cats]
            rai_means = [df_plot[(df_plot['dm_cat_int'] == c) & (df_plot['is_irrigated'] == 0)]['delta_et'].mean()
                         for c in present_cats]
            irr_stds  = [df_plot[(df_plot['dm_cat_int'] == c) & (df_plot['is_irrigated'] == 1)]['delta_et'].std()
                         for c in present_cats]
            rai_stds  = [df_plot[(df_plot['dm_cat_int'] == c) & (df_plot['is_irrigated'] == 0)]['delta_et'].std()
                         for c in present_cats]

            bars_irr = ax.bar(x - w/2, irr_means, width=w, color='#2166ac', alpha=0.8,
                              yerr=irr_stds, capsize=4, label='Irrigated')
            bars_rai = ax.bar(x + w/2, rai_means, width=w, color='#92c5de', alpha=0.8,
                              yerr=rai_stds, capsize=4, label='Rainfed')

            ax.axhline(0, color='k', linewidth=0.7)
            ax.axhline(IRR_THRESHOLD_MM, color='red', linestyle='--', linewidth=1,
                       label=f'Irrigation threshold ({IRR_THRESHOLD_MM} mm/mo)')
            ax.set_xticks(x)
            ax.set_xticklabels(cat_labels, fontsize=11)
            ax.set_xlabel('Drought category (SPEI-90d derived)', fontsize=11)
            ax.set_ylabel('HumanET (mm/month)', fontsize=11)
            ax.set_title('Mean HumanET by Drought Category: Irrigated vs Rainfed Cropland', fontsize=13)
            ax.legend(fontsize=10)

            plt.tight_layout()
            plt.savefig(figs / 'conus_et_by_drought.png', dpi=150, bbox_inches='tight')
            plt.show()
            print('Saved conus_et_by_drought.png')
    except Exception as e:
        print('ET by drought plot skipped:', e)

Saved conus_et_by_drought.png


In [21]:
if True:
    try:
        if len(df_combined) == 0:
            print("No combined data for heatmap.")
        else:
            from matplotlib.colors import TwoSlopeNorm

            # Restrict to SPEI <= 0 (drought and near-normal); all pixels, no subsampling
            df_hm = df_combined[df_combined["spei90d"] <= 0.0].dropna(
                subset=["delta_et", "spei90d", "sif_z"]
            ).copy()
            print("Observations (SPEI <= 0, complete cases):", len(df_hm))

            # 5 SPEI bins: x-axis left=near-normal, right=extreme drought
            spei_edges = [-2.5, -2.0, -1.5, -1.0, -0.5, 0.0]
            spei_labels_ordered = [
                "D4  (< -2.0)",
                "D3  (-2.0 to -1.5)",
                "D2  (-1.5 to -1.0)",
                "D1  (-1.0 to -0.5)",
                "D0  (-0.5 to 0)",
            ]
            # D0 on left (least dry) -> D4 on right (driest)
            spei_display = list(reversed(spei_labels_ordered))

            # 5 HumanET bins: y-axis, low at bottom, high at top
            det_edges  = [-200, 5, 20, 35, 60, 300]
            det_labels = [
                "< 5 mm  (very low)",
                "5-20 mm  (low)",
                "20-35 mm  (moderate)",
                "35-60 mm  (high)",
                "> 60 mm  (very high)",
            ]

            df_hm["spei_bin"] = pd.cut(df_hm["spei90d"],  bins=spei_edges,  labels=spei_labels_ordered)
            df_hm["det_bin"]  = pd.cut(df_hm["delta_et"], bins=det_edges,   labels=det_labels)

            pivot_sif = df_hm.pivot_table(values="sif_z", index="det_bin", columns="spei_bin", aggfunc="mean")
            pivot_n   = df_hm.pivot_table(values="sif_z", index="det_bin", columns="spei_bin", aggfunc="count")

            # Columns: D0 (left, least dry) -> D4 (right, driest)
            pivot_sif = pivot_sif[spei_display]
            pivot_n   = pivot_n[spei_display]

            # Rows: high HumanET at top
            pivot_sif = pivot_sif.iloc[::-1]
            pivot_n   = pivot_n.iloc[::-1]

            sif_arr    = pivot_sif.values.astype(float)
            n_arr      = pivot_n.values.astype(float)
            sif_masked = np.ma.masked_where(~np.isfinite(sif_arr), sif_arr)

            fig, ax = plt.subplots(figsize=(13, 7))

            norm = TwoSlopeNorm(vmin=-1.5, vcenter=0.0, vmax=1.5)
            im = ax.imshow(sif_masked, cmap="RdYlGn", norm=norm, aspect="auto", interpolation="nearest")

            ax.set_xticks(range(len(spei_display)))
            ax.set_xticklabels(spei_display, fontsize=11)
            ax.set_yticks(range(len(det_labels)))
            ax.set_yticklabels(list(reversed(det_labels)), fontsize=11)
            ax.set_xlabel("Drought Severity  (SPEI-90d  |  increasing drought →)", fontsize=12)
            ax.set_ylabel("HumanET / ΔET  (mm / month)", fontsize=12)
            ax.set_title(
                "Mean SIF z-score by Irrigation Intensity and Drought Severity\n"
                "CONUS Cropland, Growing Season (Apr-Sep) 2015-2024  |  SPEI ≤ 0 only",
                fontsize=13
            )

            cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
            cbar.set_label("Mean SIF z-score", fontsize=11)

            # Annotate each cell: SIF z-score on top, n= count below it
            for r in range(sif_arr.shape[0]):
                for c in range(sif_arr.shape[1]):
                    val = sif_arr[r, c]
                    n   = n_arr[r, c]
                    if np.isfinite(val):
                        txt_color = "white" if abs(val) > 0.85 else "black"
                        ax.text(c, r - 0.14, "{:.2f}".format(val),
                                ha="center", va="center",
                                fontsize=13, fontweight="bold", color=txt_color)
                        n_label = ("n=" + "{:,}".format(int(n))) if np.isfinite(n) else "n=—"
                        ax.text(c, r + 0.22, n_label,
                                ha="center", va="center", fontsize=9, color=txt_color)

            plt.tight_layout()
            plt.savefig(figs / "sif_irrigation_drought_heatmap.png", dpi=150, bbox_inches="tight")
            plt.show()
            print("Saved: sif_irrigation_drought_heatmap.png")

    except Exception as e:
        import traceback
        print("Heatmap skipped:", e)
        traceback.print_exc()


Observations (SPEI <= 0, complete cases): 208413
Saved: sif_irrigation_drought_heatmap.png


In [22]:
if True:
    try:
        # Buffering effect: mean(sif_z | D2+, irrigated) - mean(sif_z | D2+, rainfed) per pixel
        # Work pixel-by-pixel using the full (unsampled) maps to get spatial resolution

        d2plus_months = [(year, month)
                         for year in YEARS for month in GROWING_SEASON
                         if (year, month) in drought_maps and (year, month) in delta_maps]

        irr_sif_accum  = np.zeros((n_lat, n_lon), dtype=float)
        rai_sif_accum  = np.zeros((n_lat, n_lon), dtype=float)
        irr_count      = np.zeros((n_lat, n_lon), dtype=float)
        rai_count      = np.zeros((n_lat, n_lon), dtype=float)

        for (year, month) in d2plus_months:
            yyyymm = f'{year}{month:02d}'
            dm_arr = drought_maps[(year, month)]['dm_cat']
            is_d2plus = (dm_arr >= 2)

            delta_arr = delta_maps[(year, month)]
            is_irr    = (delta_arr > IRR_THRESHOLD_MM)
            is_rai    = (~is_irr) & (~np.isnan(delta_arr))

            sif_z_arr = None
            for half in ['a', 'b', '']:
                key = (yyyymm, half)
                if key in sif_zscore_maps:
                    sif_z_arr = sif_zscore_maps[key]
                    break
            if sif_z_arr is None:
                continue

            valid_sif = ~np.isnan(sif_z_arr)
            in_crop   = crop_mask_static

            mask_irr = is_d2plus & is_irr & valid_sif & in_crop
            mask_rai = is_d2plus & is_rai & valid_sif & in_crop

            irr_sif_accum[mask_irr] += sif_z_arr[mask_irr]
            irr_count[mask_irr]     += 1
            rai_sif_accum[mask_rai] += sif_z_arr[mask_rai]
            rai_count[mask_rai]     += 1

        with np.errstate(invalid='ignore', divide='ignore'):
            irr_mean = np.where(irr_count > 0, irr_sif_accum / irr_count, np.nan)
            rai_mean = np.where(rai_count > 0, rai_sif_accum / rai_count, np.nan)
            buffering_gap = irr_mean - rai_mean

        buffering_gap_crop = np.where(crop_mask_static, buffering_gap, np.nan)

        print('Buffering gap (irrigated - rainfed SIF z-score under D2+):')
        print('  Mean:    ' + str(round(float(np.nanmean(buffering_gap_crop)), 3)))
        print('  Median:  ' + str(round(float(np.nanmedian(buffering_gap_crop)), 3)))
        print('  Std:     ' + str(round(float(np.nanstd(buffering_gap_crop)), 3)))

        # Plot
        fig, ax = plt.subplots(figsize=(16, 7))
        vmax_bg = np.nanpercentile(np.abs(buffering_gap_crop), 95)
        vmax_bg = max(vmax_bg, 0.1)

        im = ax.imshow(
            buffering_gap_crop,
            extent=[CONUS_LON[0], CONUS_LON[-1], CONUS_LAT[-1], CONUS_LAT[0]],
            origin='upper', cmap='RdBu', vmin=-vmax_bg, vmax=vmax_bg, aspect='auto'
        )
        cbar = plt.colorbar(im, ax=ax, label='SIF z-score gap (irrigated \u2212 rainfed)', fraction=0.025)
        ax.set_title('Irrigation Buffering Effect under Drought (D2+): SIF z-score gap\n'
                     'Positive = irrigated crops maintain higher SIF under drought\n'
                     'CONUS Cropland, Growing Season 2015\u20132024', fontsize=12)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')

        plt.tight_layout()
        plt.savefig(figs / 'conus_sif_buffering_gap_map.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved conus_sif_buffering_gap_map.png')
    except Exception as e:
        print('Buffering gap map skipped:', e)

Buffering gap (irrigated - rainfed SIF z-score under D2+):
  Mean:    0.205
  Median:  0.238
  Std:     1.012
Saved conus_sif_buffering_gap_map.png


## 7. Summary

In [23]:
print('=' * 60)
print('CONUS HumanET x SIF x Drought Analysis -- Summary')
print('=' * 60)

# Data coverage
print()
print('--- Data Coverage ---')
print('Years analyzed: 2015-2024 (' + str(len(YEARS)) + ' years)')
et_avail  = df_et[df_et['n_cells'] > 0]
sif_avail = df_sif if len(df_sif) > 0 else pd.DataFrame()
dr_avail  = df_drought if len(df_drought) > 0 else pd.DataFrame()
print('HumanET months available:  ' + str(len(et_avail)) + ' / 120')
print('SIF records (GS only):     ' + str(len(sif_avail)))
print('GRIDMET drought months:    ' + str(len(dr_avail)))
print('USDM months:               ' + str(len(df_usdm)))

# Cropland fraction
print()
print('--- Cropland Coverage ---')
pct_crop = 100 * crop_mask_static.sum() / crop_mask_static.size
print('CONUS cells >=50% cropland: ' + str(crop_mask_static.sum()) +
      ' (' + str(round(pct_crop, 1)) + '% of CONUS grid)')

# Irrigation prevalence
print()
print('--- Irrigation Prevalence ---')
gs_et = df_et[df_et['month'].isin(GROWING_SEASON)]
if len(gs_et) > 0 and gs_et['irr_frac'].notna().any():
    mean_irr_frac = gs_et['irr_frac'].mean()
    print('Mean irrigated cell fraction (GS): ' + str(round(100 * mean_irr_frac, 1)) + '%')
    print('Mean HumanET growing season:       ' + str(round(gs_et['mean_delta'].mean(), 2)) + ' mm/mo')

# SIF anomaly stats by drought category
print()
print('--- SIF z-score by Drought Category ---')
if len(df_combined) > 0 and df_combined['sif_z'].notna().any():
    df_summary = df_combined.dropna(subset=['sif_z', 'dm_cat_int'])
    for c in sorted(df_summary['dm_cat_int'].dropna().unique()):
        sub   = df_summary[df_summary['dm_cat_int'] == c]['sif_z']
        label = DM_LABELS.get(int(c), str(c))
        print('  ' + str(int(c)) + ' (' + label + '): mean=' +
              str(round(sub.mean(), 3)) + '  std=' + str(round(sub.std(), 3)) +
              '  n=' + str(len(sub)))
else:
    print('  No combined SIF-drought data available.')

# Key drought years
print()
print('--- Top 3 Drought Years (cropland-weighted SPEI-90d severity) ---')
if len(df_drought) > 0:
    df_drought_gs = df_drought[df_drought['month'].isin(GROWING_SEASON)].copy()
    if len(df_drought_gs) > 0:
        year_severity = df_drought_gs.groupby('year')['mean_spei90d'].mean().sort_values()
        for yr_rank, (yr, val) in enumerate(year_severity.head(3).items()):
            print('  #' + str(yr_rank+1) + ': ' + str(yr) +
                  '  mean SPEI-90d = ' + str(round(val, 3)))
    else:
        print('  No growing-season drought data.')
else:
    print('  No drought data available.')

print()
print('Figures saved to:', figs)
print('Analysis complete.')

CONUS HumanET x SIF x Drought Analysis -- Summary

--- Data Coverage ---
Years analyzed: 2015-2024 (10 years)
HumanET months available:  120 / 120
SIF records (GS only):     117
GRIDMET drought months:    120
USDM months:               0

--- Cropland Coverage ---
CONUS cells >=50% cropland: 25153 (40.9% of CONUS grid)

--- Irrigation Prevalence ---
Mean irrigated cell fraction (GS): 47.4%
Mean HumanET growing season:       22.35 mm/mo

--- SIF z-score by Drought Category ---
  -1 (No drought): mean=-0.021  std=0.983  n=665454
  0 (D0): mean=-0.211  std=0.93  n=75518
  1 (D1): mean=-0.297  std=0.906  n=42797
  2 (D2): mean=-0.403  std=0.873  n=15073
  3 (D3): mean=-0.471  std=0.842  n=1353

--- Top 3 Drought Years (cropland-weighted SPEI-90d severity) ---
  #1: 2023  mean SPEI-90d = -0.081
  #2: 2021  mean SPEI-90d = -0.046
  #3: 2020  mean SPEI-90d = -0.04

Figures saved to: /home/pielab-sandbox-jcoldiron/SIF-Analysis/figures/conus
Analysis complete.
